# Mini-Project 1: Weather Forecasting

In [1]:
from pathlib import Path
import os,sys,json
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'src').exists() and (ROOT.parent/'src').exists(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.weather_client import *
LOCATION_NAME="Dresden"
CONTACT_EMAIL=os.getenv("WEATHER_CONTACT_EMAIL","B28760@students.ucu.ac.ug")
USER_AGENT=f"Advent2026WeatherNotebook/1.0 ({CONTACT_EMAIL})"
os.environ["WEATHER_USER_AGENT"]=USER_AGENT
print("Location:",LOCATION_NAME,"; replace email placeholder before MET Norway calls.")

ModuleNotFoundError: No module named 'pandas'

HTTP F

In [ ]:
try:
    urllib_candidates=geocode_urllib(LOCATION_NAME)
    print("urllib candidates:",len(urllib_candidates))
except WeatherAPIError as e:
    urllib_candidates=[]; print("urllib unavailable:",e)
try:
    requests_candidates=geocode(LOCATION_NAME)
    print("requests candidates:",len(requests_candidates))
    _,meta=request_json("https://geocoding-api.open-meteo.com/v1/search",
       {"name":LOCATION_NAME,"count":1,"language":"en","format":"json"})
    print("Status, Content-Type, elapsed seconds:",meta["status_code"],meta["content_type"],meta["elapsed_seconds"])
except WeatherAPIError as e:
    requests_candidates=[]; print("requests unavailable:",e)

## FORECAST AND RETRIEVAL DISPLAY

In [ ]:
try:
    candidates=geocode(LOCATION_NAME)
    if not candidates:
        print("No geocoding results; showing saved illustrative sample.")
        payload=json.loads((ROOT/'data'/'sample_forecast.json').read_text())
        forecast_df=pd.DataFrame(forecast_to_records(payload))
    else:
        display(pd.DataFrame([{"choice":i,"name":c.get("name"),"country":c.get("country"),
          "admin":c.get("admin1"),"latitude":c.get("latitude"),"longitude":c.get("longitude")}
          for i,c in enumerate(candidates)]))
        selected=select_location(candidates,0)
        payload=get_forecast(selected["latitude"],selected["longitude"],days=7)
        forecast_df=pd.DataFrame(forecast_to_records(payload))
        print("Selected:",selected.get("name"),selected.get("country"),"Source: Open-Meteo")
        display(forecast_df)
except (WeatherAPIError,ValueError,KeyError) as e:
    payload=json.loads((ROOT/'data'/'sample_forecast.json').read_text())
    forecast_df=pd.DataFrame(forecast_to_records(payload))
    print("Live API unavailable; showing illustrative saved sample instead:",e)
    display(forecast_df)

In [ ]:
if not forecast_df.empty:
    x=pd.to_datetime(forecast_df["Date"])
    fig,ax=plt.subplots(figsize=(11,5))
    ax.fill_between(x,forecast_df["Minimum temperature (°C)"],forecast_df["Maximum temperature (°C)"],alpha=.25,label="Temperature range")
    ax.plot(x,forecast_df["Maximum temperature (°C)"],marker="o",label="Maximum °C")
    ax.plot(x,forecast_df["Minimum temperature (°C)"],marker="o",label="Minimum °C")
    ax.set_ylabel("Temperature (°C)"); ax.set_xlabel("Local date")
    ax2=ax.twinx(); ax2.bar(x,forecast_df["Precipitation (mm)"],alpha=.25,label="Precipitation")
    ax2.set_ylabel("Precipitation (mm)"); ax.set_title("Daily temperature range and precipitation")
    ax.legend(loc="upper left"); ax2.legend(loc="upper right"); ax.grid(alpha=.2); plt.show()
    print("Mean maximum temperature:",round(forecast_df["Maximum temperature (°C)"].mean(),1),"°C")
    print("Total precipitation:",round(forecast_df["Precipitation (mm)"].sum(),1),"mm")

 ## robustness and second source

In [ ]:
for label,action in [
 ("invalid latitude",lambda:get_forecast(100,0)),
 ("simulated offline network",lambda:request_json("http://127.0.0.1:1/not-running",timeout=.2,retries=2)),
 ("tiny timeout",lambda:request_json("https://geocoding-api.open-meteo.com/v1/search",{"name":LOCATION_NAME},timeout=.001,retries=1))]:
    try: action()
    except (ValueError,WeatherAPIError) as e: print(label,"handled:",e)
try:
    missing=geocode("this-place-name-should-not-exist-xyz",timeout=3)
    print("Invalid city candidates:",len(missing))
except WeatherAPIError as e: print("Invalid city/network issue handled:",e)
if "your-university-email@example.com" not in CONTACT_EMAIL and "@" in CONTACT_EMAIL:
    try: print("MET Norway keys:",list(met_norway_forecast(selected["latitude"],selected["longitude"],USER_AGENT).keys()))
    except (WeatherAPIError,ValueError,NameError) as e: print("Second source unavailable:",e)
else: print("MET Norway call skipped until real university email is configured.")

## Report

### Data flow
Place name input → Open-Meteo geocoding → candidate list → selected latitude/longitude → forecast API → JSON parsing and WMO code decoding → pandas table → summary and chart.

### Limitations and conclusion
Weather forecasts change and are probabilistic. Network availability, rate limits and API schema changes remain external dependencies. The fixture validates parsing but does not prove current conditions. A real product should cache responsibly, monitor provider changes, retain source attribution and validate dates and units against current API documentation.